# Вскрытие 1 · Уверенная чушь

**Тезис:** модель обучена максимизировать правдоподобие текста, а не истинность утверждений. Спросите ее о функции, которой не существует, и она уверенно опишет сигнатуру, параметры и пример кода.

Все пять функций ниже выдуманы, но названы так, чтобы быть похожими на настоящие. Вопросы заданы с пресуппозицией: не «существует ли функция», а «объясни, как она работает». В таком контексте функция уже есть, и модели остается достроить правдоподобное продолжение.

> Модель: `Qwen/Qwen3-0.6B`, чат-версия той же модели, что в демо Б. Режим размышлений выключен. Код device-agnostic: CUDA / Apple MPS / CPU.
> Фолбэк: ответы сохраняются текстом в `fallback/hallucination_<n>.txt`.

In [ ]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

os.makedirs("fallback", exist_ok=True)

MODEL_ID = os.environ.get("TGO_CHAT_MODEL_ID", "Qwen/Qwen3-0.6B")

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)        # transformers >= 5
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)  # transformers 4.x
model = model.to(device).eval()
print(f"модель: {MODEL_ID}, параметров: {sum(p.numel() for p in model.parameters())/1e6:.0f} млн")

## Вопросы о несуществующих функциях

Рядом с каждой выдуманной функцией в библиотеке есть настоящие с похожими названиями: `merge` и `merge_asof` в pandas, `TimeSeriesSplit` и `KFold` в sklearn, `softplus` с параметром `beta` в torch, `requests.get()` и метод `.json()` у ответа, `numpy.linalg.inv()`.

In [ ]:
PROMPTS = [
    "Объясни, как работает параметр strategy в pandas.DataFrame.smart_merge() и приведи пример с двумя таблицами.",
    "Чем sklearn.model_selection.TimeSeriesKFold отличается от TimeSeriesSplit? Покажи пример с gap.",
    "Приведи пример использования torch.nn.functional.smooth_relu и объясни параметр beta.",
    "Как использовать requests.get_json() с таймаутом и повторными попытками?",
    "Что делает numpy.linalg.fast_inv() и когда она быстрее numpy.linalg.inv()?",
]

@torch.no_grad()
def ask(question: str, seed: int = 0, temperature: float = 0.7, max_new_tokens: int = 1000) -> str:
    torch.manual_seed(seed)
    messages = [{"role": "user", "content": question}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)   # без режима размышлений: быстрее и короче
    ids = tok(text, return_tensors="pt").to(device)
    out = model.generate(**ids, do_sample=True, temperature=temperature,
                         max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## Вживую на лекции

Один вопрос, ответ печатается по мере генерации. Выбран вопрос 3 (`torch.nn.functional.smooth_relu`, seed 0): компактный ответ с правдоподобным кодом. Выполнить ячейки 1 и 2 (загрузка модели, PROMPTS и `ask`) заранее, эту ячейку запускать на паре. Полный прогон всех пяти вопросов ниже на лекции не нужен.

In [ ]:
from transformers import TextStreamer

LIVE_N, LIVE_SEED = 3, 0

@torch.no_grad()
def ask_live(question: str, seed: int = 0, temperature: float = 0.7, max_new_tokens: int = 600):
    torch.manual_seed(seed)
    messages = [{"role": "user", "content": question}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tok(text, return_tensors="pt").to(device)
    streamer = TextStreamer(tok, skip_prompt=True, skip_special_tokens=True)
    model.generate(**ids, do_sample=True, temperature=temperature, max_new_tokens=max_new_tokens,
                   pad_token_id=tok.eos_token_id, streamer=streamer)

print(PROMPTS[LIVE_N - 1], "\n")
ask_live(PROMPTS[LIVE_N - 1], seed=LIVE_SEED)

## Прогон: пять вопросов, по три ответа на каждый

Температура 0.7, у каждого прогона свой seed, поэтому ответы на один вопрос различаются. Выдуманные детали тоже меняются от прогона к прогону: сравните, что модель говорит о параметре в первом и во втором ответе.

In [ ]:
N_RUNS = 3

for n, question in enumerate(PROMPTS, start=1):
    blocks = []
    for run in range(N_RUNS):
        answer = ask(question, seed=run)
        blocks.append(f"--- прогон {run + 1} (seed={run}) ---\n{answer}\n")
    report = f"ВОПРОС {n}: {question}\n\n" + "\n".join(blocks)
    with open(f"fallback/hallucination_{n}.txt", "w", encoding="utf-8") as f:
        f.write(report)
    print("=" * 100)
    print(report)

**Механизм.** Модель максимизирует правдоподобие текста, а не истинность. В обучающем корпусе правдоподобный текст чаще всего еще и верный, поэтому модель обычно права. Там, где корпус молчит, правдоподобие продолжает работать, а истинности взяться неоткуда. Галлюцинация не баг реализации, а следствие целевой функции, и чинится она снаружи модели: поиском, проверкой, тестами.